# ch07 训练与评估（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`data/load_curve.csv` → A 台区 8760 h；窗口 72、步长 24、起点从 168 开始、按时间 8:2 切分。
>
> **每题都跟了 `# 提示：`**，照着提示能独立做完。
> 如果 assert 的数字对不上，**先回头看上一步的输出**，别急着改 assert —— 那些数字都是实跑出来的。
> 注意：涉及 torch 训练的断言都是**区间**（不是死值），因为训练结果依赖随机数种子；
> 只有**朴素基线**和**指标函数本身**是确定性的，那几个数才能卡死。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 构造 72→24 样本 + 8:2 时间切分 | 3 | §7.1 |
| 2 | 手写 MSE / MAE / RMSE / MAPE + 对账 | 4 | §7.1 |
| 3 | MAPE 零点陷阱 + sMAPE / WAPE | 3 | §7.1.1 |
| 4 | 早停 + 梯度裁剪 + StepLR 训练循环 | 3 | §7.2 |
| 5 | 训练曲线可视化（loss / grad / lr） | 1 | §7.2.1 |
| 6 | 递归 vs 直接多输出 | 4 | §7.3 |
| 7 | 逐步 MAE 曲线 + 方差放大 | 2 | §7.3.1 |
| 8 | 预测曲线可视化（整段 + 单日） | 2 | §7.4 |
| 9 | 残差 ACF（手写 + 对账） | 2 | §7.5 |
| 10 | 按小时残差 + 显著性检验 | 2 | §7.5.1 |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA = Path("data")          # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
print("原始行数", len(raw), "| 列", list(raw.columns))
print("台区", sorted(raw["台区编号"].unique()))
print("torch", torch.__version__, "| CUDA 可用？", torch.cuda.is_available(), "（数据小，CPU 足够）")

In [ ]:
# 去重 + 排序（顺序不能反，理由见 ch01 §1.1）
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
frame = df[df["台区编号"] == "STATION_A_01"].set_index("时间戳").sort_index()


def clean_series(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值（ch01 §1.3.2 的结论）。"""
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


s = clean_series(frame["负荷值"].asfreq("h"))
ser = s.values                      # 本章统一用 ndarray，方便做窗口切片

print("A 台区 %d 个整点 | 原始缺失 %d" % (len(s), int(frame["负荷值"].asfreq("h").isna().sum())))
print("清洗后 min %.2f | max %.2f | mean %.6f" % (ser.min(), ser.max(), ser.mean()))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

In [ ]:
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容
WINDOW, HORIZON = 72, 24
# 起点取 168 而不是 WINDOW：与 ch08 对齐（ch08 要用 lag168，历史至少要 168 小时）
START = 168


class MLP(nn.Module):
    """两层 ReLU 的全连接网络。

    输入 = 展平的 72 小时窗口；输出维度由 out 决定：
    递归单步模型用 out=1，直接多步模型用 out=24（见 §7.3）。
    """

    def __init__(self, inp, hid, out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp, hid), nn.ReLU(),
            nn.Linear(hid, hid), nn.ReLU(),
            nn.Linear(hid, out),
        )

    def forward(self, x):
        return self.net(x)


def mae_by_step(y_true, y_pred):
    """返回长度 HORIZON 的逐步 MAE：第 h 个元素 = 「预测 h+1 步」的平均绝对误差。"""
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), axis=0)


def mae_all(y_true, y_pred):
    """24 步合起来的单个 MAE（后面所有对比表的排序依据）。"""
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

## 任务 1：构造 72 步窗口与 8:2 时间切分（§7.1）

In [ ]:
origins = np.arange(START, len(ser) - HORIZON + 1)

X = np.stack([ser[t - WINDOW:t] for t in origins]).astype("float32")
Y = np.stack([ser[t:t + HORIZON] for t in origins]).astype("float32")

cut = int(len(origins) * 0.8)

X_train, Y_train = X[:cut], Y[:cut]
X_test, Y_test = X[cut:], Y[cut:]
t_test = frame.index[origins[cut:]]

# 朴素基线：季节性朴素（用昨天同一小时），§7.1 拿它当「被打分的预测」
PN = np.stack([ser[t + np.arange(1, HORIZON + 1) - 24] for t in origins[cut:]])

print("样本 %d（起点 %d ~ %d）| 切分点 %d | 训练 %d / 测试 %d"
      % (len(origins), origins[0], origins[-1], cut, len(X_train), len(X_test)))
print("训练段目标均值 %.6f | 测试段目标均值 %.4f" % (Y_train.mean(), Y_test.mean()))
print("测试段时段 %s ~ %s" % (t_test[0], t_test[-1]))
print("测试段真值 min %.4f max %.4f | 季节朴素预测 min %.4f max %.4f"
      % (Y_test.min(), Y_test.max(), PN.min(), PN.max()))
print("窗口最后一个点 X_train[0, -1] = %.4f（= ser[%d]，是**已知**的）" % (X_train[0, -1], START - 1))
print("第一个待预测点 Y_train[0, 0]   = %.4f（= ser[%d]，窗口里没有它）" % (Y_train[0, 0], START))

# ---- 验收 ----
assert len(origins) == 8569 and cut == 6855
assert X_train.shape == (6855, WINDOW) and Y_test.shape == (1714, HORIZON)
assert abs(float(Y_train.mean()) - 452.674164) < 1e-5
assert abs(float(Y_test.mean()) - 483.2289) < 1e-4
assert str(t_test[0]) == "2025-10-20 15:00:00" and str(t_test[-1]) == "2025-12-31 00:00:00"
assert X_train[0, -1] == np.float32(ser[START - 1]) and Y_train[0, 0] == np.float32(ser[START])
assert Y_train[0, 23] == np.float32(ser[START + HORIZON - 1]), "第 23 列就是 h=24"

## 任务 2：手写四个指标，并与 sklearn 对账（§7.1）

In [ ]:
def mse(y_true, y_pred):
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)


MSE, MAE, RMSE, MAPE = mse(Y_test, PN), mae(Y_test, PN), rmse(Y_test, PN), mape(Y_test, PN)
print("MSE  %12.6f" % MSE)
print("MAE  %12.6f" % MAE)
print("RMSE %12.6f" % RMSE)
print("MAPE %12.6f %%" % MAPE)
print("RMSE / MAE = %.6f   ← 恒 ≥ 1，这个比值在说误差分布的形状" % (RMSE / MAE))
print("sklearn 对账：MAE %.6f | MSE %.6f" % (mean_absolute_error(Y_test, PN),
                                             mean_squared_error(Y_test, PN)))
print("手写 - sklearn：MAE %.3e | MSE %.3e"
      % (MAE - mean_absolute_error(Y_test, PN), MSE - mean_squared_error(Y_test, PN)))

# ---- 验收 ----
assert abs(MSE - 1802.744577) < 1e-5
assert abs(MAE - 32.998794) < 1e-5
assert abs(RMSE - 42.458740) < 1e-5
assert abs(MAPE - 6.693991) < 1e-5
assert RMSE >= MAE, "RMSE ≥ MAE 恒成立（Jensen 不等式）"
assert abs(RMSE / MAE - 1.286676) < 1e-5
assert abs(MAE - mean_absolute_error(Y_test, PN)) < 1e-9
assert abs(MSE - mean_squared_error(Y_test, PN)) < 1e-9

## 任务 3：MAPE 的零点陷阱与两种修法（§7.1.1）

In [ ]:
Y_trap = Y_test.copy()
Y_trap[:, :6] = Y_trap[:, :6] * 0.01


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100)


def wape(y_true, y_pred):
    #       换成「绝对误差之和 / 真值之和」—— 分母换成总量，就不会归零
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100)


print("缩放后真值 min %.6f（原 min %.4f）—— 分母塌了" % (Y_trap.min(), Y_test.min()))
print("MAE   %12.6f（原 %.6f，只涨 %.2f 倍）" % (mae(Y_trap, PN), MAE, mae(Y_trap, PN) / MAE))
print("RMSE  %12.6f（原 %.6f，涨 %.2f 倍）" % (rmse(Y_trap, PN), RMSE, rmse(Y_trap, PN) / RMSE))
print("MAPE  %12.6f（原 %.6f，涨 %.1f 倍）← 爆炸" % (mape(Y_trap, PN), MAPE, mape(Y_trap, PN) / MAPE))
print("sMAPE %12.6f（原 %.6f，涨 %.1f 倍）" % (smape(Y_trap, PN), smape(Y_test, PN),
                                              smape(Y_trap, PN) / smape(Y_test, PN)))
print("WAPE  %12.6f（原 %.6f，涨 %.1f 倍）" % (wape(Y_trap, PN), wape(Y_test, PN),
                                              wape(Y_trap, PN) / wape(Y_test, PN)))

print()
print("单点极端例：真值 0.05，预测 5.0 —— 绝对误差只有 %.2f kW" % abs(5.0 - 0.05))
print("  MAPE  %10.4f %%   ← 一个点就把整份报告毁掉" % mape(np.array([0.05]), np.array([5.0])))
print("  sMAPE %10.4f %%   ← 有上界（≤ 200）" % smape(np.array([0.05]), np.array([5.0])))

# ---- 验收 ----
assert abs(mae(Y_trap, PN) - 143.890302) < 1e-5
assert abs(rmse(Y_trap, PN) - 243.244359) < 1e-5
assert abs(mape(Y_trap, PN) - 2481.953943) < 1e-5
assert abs(smape(Y_trap, PN) - 54.016410) < 1e-5
assert abs(wape(Y_trap, PN) - 39.555869) < 1e-5
assert abs(smape(Y_test, PN) - 6.682859) < 1e-5
assert abs(wape(Y_test, PN) - 6.828811) < 1e-5
assert mape(Y_trap, PN) / MAPE > 300, "MAPE 涨了 300 倍以上，sMAPE/WAPE 只涨个位数倍"
assert smape(Y_trap, PN) / smape(Y_test, PN) < 10 and wape(Y_trap, PN) / wape(Y_test, PN) < 10
assert abs(mape(np.array([0.05]), np.array([5.0])) - 9900.0) < 1e-6
assert abs(smape(np.array([0.05]), np.array([5.0])) - 196.0396) < 1e-3

## 任务 4：训练循环模板 —— 早停 / 梯度裁剪 / lr 调度（§7.2）

In [ ]:
# 标准化参数只能来自训练段（ch01 §1.4.1 的纪律，零成本）
MU, SD = float(X_train.mean()), float(X_train.std())
MUY, SDY = float(Y_train.mean()), float(Y_train.std())

Xtr = torch.tensor((X_train - MU) / SD)
Ytr = torch.tensor((Y_train - MUY) / SDY)
Xte = torch.tensor((X_test - MU) / SD)

# 训练段内部再切一段「时间靠后」的验证集，专门给早停看（还是按时间，不 shuffle）
vcut = int(len(Xtr) * 0.85)
X_fit, Y_fit = Xtr[:vcut], Ytr[:vcut]
X_val, Y_val = Xtr[vcut:], Ytr[vcut:]

SUB, PATIENCE, MAX_EPOCH, CLIP, STEP, GAMMA = 2500, 5, 40, 1.0, 8, 0.5
print("训练子集 %d | 拟合段 %d / 验证段 %d | patience %d | 最大轮数 %d | 裁剪阈值 %.1f"
      % (SUB, len(X_fit), len(X_val), PATIENCE, MAX_EPOCH, CLIP))

with torch.random.fork_rng():
    torch.manual_seed(42)
    es_model = MLP(WINDOW, 128, 1)
    opt = torch.optim.Adam(es_model.parameters(), lr=2e-3)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=STEP, gamma=GAMMA)
    tr_hist, va_hist, gn_hist, lr_hist = [], [], [], []
    best_val, best_epoch, wait, stopped_epoch = float("inf"), -1, 0, MAX_EPOCH
    for ep in range(1, MAX_EPOCH + 1):
        es_model.train()
        opt.zero_grad()
        loss = ((es_model(X_fit[:SUB]) - Y_fit[:SUB, 0:1]) ** 2).mean()
        loss.backward()
        gnorm = float(torch.nn.utils.clip_grad_norm_(es_model.parameters(), CLIP))
        opt.step()
        sched.step()
        es_model.eval()
        with torch.no_grad():
            val_loss = float(((es_model(X_val) - Y_val[:, 0:1]) ** 2).mean())
        tr_hist.append(float(loss.detach()))
        va_hist.append(val_loss)
        gn_hist.append(gnorm)
        lr_hist.append(opt.param_groups[0]["lr"])
        # 早停计数：验证损失不再创新低就 wait+1，连续 PATIENCE 轮就停
        if val_loss < best_val - 1e-9:
            best_val, best_epoch, wait = val_loss, ep, 0
        else:
            wait = wait + 1
            if wait >= PATIENCE:
                stopped_epoch = ep
                break

print("实际跑了 %d 轮 | 最好的一轮 best_epoch = %d（val %.6f）| 停在第 %d 轮"
      % (len(tr_hist), best_epoch, best_val, stopped_epoch))
print("训练损失 %.4f → %.4f（还在降）| 验证损失 %.4f → %.4f（先降后升）"
      % (tr_hist[0], tr_hist[-1], va_hist[0], va_hist[-1]))
print("梯度范数 %.3f → %.3f | 最大 %.3f（裁剪阈值 %.1f）| 被裁过的轮数 %d"
      % (gn_hist[0], gn_hist[-1], max(gn_hist), CLIP, sum(1 for g in gn_hist if g > CLIP)))
print("学习率 %.6f → %.6f（StepLR：每 %d 轮 ×%.1f）"
      % (lr_hist[0], lr_hist[-1], STEP, GAMMA))

# ---- 验收 ----
assert stopped_epoch < MAX_EPOCH, "早停必须真的触发，否则这个演示没有意义"
assert stopped_epoch == best_epoch + PATIENCE, "停下来的那一轮 = 最好的一轮 + patience"
assert 3 <= best_epoch <= 25 and len(tr_hist) == stopped_epoch
assert va_hist[best_epoch - 1] == min(va_hist), "best_epoch 就是验证损失最小那一轮"
assert tr_hist[-1] < tr_hist[0], "训练损失还在降 —— 典型的过拟合形状"
assert max(gn_hist) > CLIP, "至少有一轮的梯度范数超过阈值，裁剪才是必要的"
assert abs(lr_hist[0] - 2e-3) < 1e-18
assert abs(lr_hist[-1] - 2e-3 * GAMMA ** (stopped_epoch // STEP)) < 1e-18

## 任务 5：把训练过程画出来（§7.2.1）

In [ ]:
eps = np.arange(1, len(tr_hist) + 1)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
axes[0].plot(eps, tr_hist, marker="o", ms=3, label="训练损失（子集）")
axes[0].plot(eps, va_hist, marker="s", ms=3, label="验证损失")
axes[0].axvline(best_epoch, color="seagreen", ls="--", lw=1.2, label="best_epoch=%d" % best_epoch)
axes[0].axvline(stopped_epoch, color="crimson", ls=":", lw=1.2, label="stopped_epoch=%d" % stopped_epoch)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("MSE（标准化后）")
axes[0].set_title("早停：验证损失先降后升")
axes[0].legend(fontsize=8)

axes[1].plot(eps, gn_hist, marker="o", ms=3, color="tab:orange", label="梯度范数（裁剪前）")
axes[1].axhline(CLIP, color="crimson", ls="--", lw=1.2, label="clip = %.1f" % CLIP)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("‖grad‖")
axes[1].set_title("梯度裁剪")
axes[1].legend(fontsize=8)

axes[2].plot(eps, lr_hist, marker="o", ms=3, color="tab:green")
axes[2].set_yscale("log")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("学习率")
axes[2].set_title("StepLR：每 %d 轮减半" % STEP)

plt.tight_layout()
print("三个量都能从历史列表里还原：loss %d / grad %d / lr %d 个点"
      % (len(tr_hist), len(gn_hist), len(lr_hist)))

# ---- 验收 ----
assert len(eps) == stopped_epoch
assert int(np.argmin(va_hist)) + 1 == best_epoch
assert lr_hist[-1] <= lr_hist[0] and len(set(lr_hist)) <= 3

## 任务 6：递归预测 vs 直接多输出（§7.3）

In [ ]:
with torch.random.fork_rng():
    torch.manual_seed(7)
    rec_model = MLP(WINDOW, 128, 1)
    rec_opt = torch.optim.Adam(rec_model.parameters(), lr=1e-2)
    for _ in range(40):
        rec_opt.zero_grad()
        ((rec_model(Xtr) - Ytr[:, 0:1]) ** 2).mean().backward()
        rec_opt.step()
    rec_model.eval()

    buf = ((X_test - MU) / SD).copy()
    rec_pred = np.zeros_like(Y_test)
    with torch.no_grad():
        for h in range(HORIZON):
            p = rec_model(torch.tensor(buf)).numpy().ravel() * SDY + MUY
            rec_pred[:, h] = p
            buf = np.concatenate([buf[:, 1:], ((p - MUY) / SD)[:, None]], axis=1)

print("递归模型 24 步 MAE %.6f（h=1 %.4f → h=24 %.4f）"
      % (mae_all(Y_test, rec_pred), mae_by_step(Y_test, rec_pred)[0], mae_by_step(Y_test, rec_pred)[23]))
print("递归预测的均值 %.4f / std %.4f | 真值均值 %.4f / std %.4f"
      % (rec_pred.mean(), rec_pred.std(), Y_test.mean(), Y_test.std()))

with torch.random.fork_rng():
    torch.manual_seed(7)
    dir_model = MLP(WINDOW, 128, HORIZON)
    dir_opt = torch.optim.Adam(dir_model.parameters(), lr=1e-2)
    for _ in range(40):
        dir_opt.zero_grad()
        ((dir_model(Xtr) - Ytr) ** 2).mean().backward()
        dir_opt.step()
    dir_model.eval()
    with torch.no_grad():
        dir_pred = dir_model(Xte).numpy() * SDY + MUY

print("直接多输出 24 步 MAE %.6f（h=1 %.4f → h=24 %.4f）"
      % (mae_all(Y_test, dir_pred), mae_by_step(Y_test, dir_pred)[0], mae_by_step(Y_test, dir_pred)[23]))
print("两个模型的 MAE 都远好于季节朴素基线 %.6f" % mae_all(Y_test, PN))

# ---- 验收 ----
assert rec_pred.shape == dir_pred.shape == Y_test.shape == (1714, HORIZON)
assert 10 < mae_all(Y_test, rec_pred) < 25, "递归模型应在「远好于基线」的量级，但不该是死值"
assert 10 < mae_all(Y_test, dir_pred) < 25
assert mae_all(Y_test, rec_pred) < mae_all(Y_test, PN)
assert mae_all(Y_test, dir_pred) < mae_all(Y_test, PN)
assert abs(mae_all(Y_test, PN) - 32.998794) < 1e-5, "基线是确定性的，可以卡死值"
assert rec_pred.mean() > 0 and rec_pred.std() > 0

## 任务 7：多步误差累积曲线（§7.3.1）

In [ ]:
step_rec = mae_by_step(Y_test, rec_pred)
step_dir = mae_by_step(Y_test, dir_pred)
step_pn = mae_by_step(Y_test, PN)

print("h      递归      直接      季节朴素")
for h in range(HORIZON):
    print("%2d  %8.4f  %8.4f  %8.4f" % (h + 1, step_rec[h], step_dir[h], step_pn[h]))
print("整体 %8.4f  %8.4f  %8.4f" % (step_rec.mean(), step_dir.mean(), step_pn.mean()))

err_rec = np.abs(Y_test - rec_pred)
print()
print("递归误差的**标准差**（不是 MAE）：h1 %.4f → h8 %.4f → h24 %.4f（涨 %.2f 倍）"
      % (err_rec[:, 0].std(), err_rec[:, 7].std(), err_rec[:, 23].std(),
         err_rec[:, 23].std() / err_rec[:, 0].std()))

h_grid = np.arange(1, HORIZON + 1)
std_by_step = err_rec.std(axis=0)
b_exp = float(np.polyfit(np.log(h_grid), np.log(std_by_step), 1)[0])

print("幂律拟合 std(h) ∝ h^%.4f  ← 若是「独立误差线性累加」b 会接近 0.5" % b_exp)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
axes[0].plot(h_grid, step_rec, marker="o", ms=3, label="递归（自己喂自己）")
axes[0].plot(h_grid, step_dir, marker="s", ms=3, label="直接多输出")
axes[0].plot(h_grid, step_pn, marker="^", ms=3, label="季节朴素 t-24")
axes[0].set_xlabel("预测步长 h（小时）")
axes[0].set_ylabel("MAE (kW)")
axes[0].set_title("多步误差累积：递归曲线单调上翘")
axes[0].legend(fontsize=9)

axes[1].plot(h_grid, std_by_step, marker="o", ms=3, color="crimson", label="递归误差 std")
axes[1].plot(h_grid, std_by_step[0] * h_grid ** b_exp, ls="--", lw=1.2,
             label="幂律拟合 h^%.3f" % b_exp)
axes[1].set_xlabel("预测步长 h（小时）")
axes[1].set_ylabel("误差标准差 (kW)")
axes[1].set_title("方差随时间放大")
axes[1].legend(fontsize=9)
plt.tight_layout()

# ---- 验收 ----
assert step_rec.shape == step_dir.shape == step_pn.shape == (HORIZON,)
assert abs(step_pn[0] - 33.0215) < 1e-4 and abs(step_pn[23] - 33.1928) < 1e-4
assert step_rec[-1] > step_rec[0] * 1.15, "递归误差随步长明显上翘"
assert step_dir[-1] > step_dir[0], "直接多输出也会涨，但通常更平"
assert err_rec[:, 23].std() > err_rec[:, 0].std() * 1.15, "误差方差随时间放大"
assert 0.0 < b_exp < 0.6, "放大速度介于「完全独立」（0.5）与「几乎不涨」（0）之间"

## 任务 8：预测曲线可视化（§7.4）

In [ ]:
show = np.arange(0, len(Y_test), 4)

k = 369                      # 单日放大的起点（测试段内第 370 个窗口）
day_idx = np.arange(k, k + HORIZON)
print("整段抽样 %d 个点 | 单日放大起点 %s（h=1 对应 %s）"
      % (len(show), t_test[k], t_test[k]))

fig, axes = plt.subplots(2, 1, figsize=(13, 7.2),
                         gridspec_kw={"height_ratios": [1.35, 1]})
axes[0].plot(t_test[show], Y_test[show, 0], lw=1.0, label="真实（h=1 的时刻值）")
axes[0].plot(t_test[show], rec_pred[show, 0], lw=1.0, label="递归模型 1 步预测")
axes[0].set_ylabel("负荷 kW")
axes[0].set_title("整段抽样：测试段 1714 个起点里每 4 个取 1 个")
axes[0].legend(fontsize=9)

axes[1].plot(np.arange(1, HORIZON + 1), Y_test[k], marker="o", ms=4, color="black", label="真实")
axes[1].plot(np.arange(1, HORIZON + 1), rec_pred[k], marker="s", ms=3, label="递归")
axes[1].plot(np.arange(1, HORIZON + 1), dir_pred[k], marker="^", ms=3, label="直接多输出")
axes[1].plot(np.arange(1, HORIZON + 1), PN[k], marker="x", ms=3, label="季节朴素 t-24")

axes[1].set_xlabel("预测步长 h（小时）")
axes[1].set_ylabel("负荷 kW")
axes[1].set_title("单日放大：起点 %s 的 24 步预测" % t_test[k])
axes[1].legend(fontsize=9)
plt.tight_layout()

# ---- 验收 ----
assert len(show) == 429
assert abs(float(Y_test[k, 0]) - float(ser[origins[cut] + k])) < 1e-3, "窗口目标与原始序列对齐"
assert str(t_test[k]) == "2025-11-05 00:00:00"
assert Y_test[k].shape == rec_pred[k].shape == dir_pred[k].shape == PN[k].shape == (HORIZON,)

## 任务 9：残差自相关（§7.5）

In [ ]:
res1 = rec_pred[:, 0] - Y_test[:, 0]
res24 = rec_pred[:, 23] - Y_test[:, 23]


def acf(x, lag):
    x = np.asarray(x, dtype=float) - np.mean(x)
    return float(np.sum(x[:-lag] * x[lag:]) / np.sum(x * x))


LAGS = (1, 2, 3, 24, 168)
acf_r1 = [round(acf(res1, k), 4) for k in LAGS]
acf_r24 = [round(acf(res24, k), 4) for k in LAGS]

naive1 = np.array([ser[t - 24] for t in origins[cut:]]) - Y_test[:, 0]

print("lag      " + "  ".join("lag%-6d" % k for k in LAGS))
print("h=1  残差 " + "  ".join("%+.4f " % v for v in acf_r1))
print("h=24 残差 " + "  ".join("%+.4f " % v for v in acf_r24))
print("朴素 残差 " + "  ".join("%+.4f " % acf(naive1, k) for k in LAGS))
print()
print("手写 acf vs pandas .autocorr（同一份残差，两种口径）：")
print("  lag=1  手写 %+.6f | pandas %+.6f | 差 %+.6f"
      % (acf(res1, 1), pd.Series(res1).autocorr(1), acf(res1, 1) - pd.Series(res1).autocorr(1)))
print("  lag=24 手写 %+.6f | pandas %+.6f | 差 %+.6f"
      % (acf(res1, 24), pd.Series(res1).autocorr(24),
         acf(res1, 24) - pd.Series(res1).autocorr(24)))
print("  （差别来自分母：手写用全序列方差，pandas 用两个错位子序列各自的 std）")

# ---- 验收 ----
assert len(acf_r1) == len(acf_r24) == 5
assert -0.3 < acf_r1[0] < 0.4, "h=1 残差的自相关很弱 —— 模型把日周期吃掉了"
assert acf_r24[0] > acf_r1[0] + 0.2, "多步残差仍有强自相关：误差是「成串」的"
assert abs(acf(naive1, 1) - 0.3804) < 1e-4, "朴素残差是确定性的，可以卡死值"
assert abs(acf(naive1, 24) - (-0.3149)) < 1e-4
assert abs(acf(res1, 1) - pd.Series(res1).autocorr(1)) < 0.05
try:
    acf(res1, 0)
    raise AssertionError("lag=0 应该报错")
except ValueError:
    print("lag=0 → ValueError（x[:-0] 是空数组）—— 手写 ACF 的边界坑")

## 任务 10：按小时分组的残差与显著性（§7.5.1）

In [ ]:
res_frame = pd.DataFrame({"hour": t_test.hour, "res": res1})
by_hour = res_frame.groupby("hour")["res"]
hour_mean, hour_std = by_hour.mean(), by_hour.std()

t1 = float(res1.mean() / (res1.std(ddof=1) / np.sqrt(len(res1))))
t24 = float(res24.mean() / (res24.std(ddof=1) / np.sqrt(len(res24))))

for nm, r, t in [("h=1 ", res1, t1), ("h=24", res24, t24)]:
    print("%s 残差均值 %+8.4f kW（占负荷均值 %+.4f%%）| std %7.4f | t = %+.4f"
          % (nm, r.mean(), r.mean() / Y_test.mean() * 100, r.std(), t))
print("临界值：n=%d 时 |t| > 1.96 就算「在 5%% 水平上显著」" % len(res1))
print()
print("按小时的残差均值：最低 %+.4f kW（%d 点）| 最高 %+.4f kW（%d 点）| 极差 %.4f"
      % (hour_mean.min(), int(hour_mean.idxmin()), hour_mean.max(), int(hour_mean.idxmax()),
         hour_mean.max() - hour_mean.min()))
print("按小时的残差标准差：最稳 %d 点 %.4f | 最不稳 %d 点 %.4f"
      % (int(hour_std.idxmin()), hour_std.min(), int(hour_std.idxmax()), hour_std.max()))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
axes[0].boxplot([res_frame.loc[res_frame["hour"] == h, "res"].values for h in range(24)],
                tick_labels=list(range(24)), showfliers=False)
axes[0].axhline(0, color="black", lw=1)
axes[0].set_xlabel("预测时刻（小时）")
axes[0].set_ylabel("残差 kW")
axes[0].set_title("残差按小时分组：晚间波动更大")
axes[1].bar(hour_mean.index, hour_mean.values, color="tab:blue")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("预测时刻（小时）")
axes[1].set_ylabel("残差均值 kW")
axes[1].set_title("系统性偏差：午后高估、清晨低估")
plt.tight_layout()

# ---- 验收 ----
assert len(hour_mean) == 24 and len(hour_std) == 24
assert int(by_hour.size().sum()) == len(res1) == 1714
assert hour_std.max() > hour_std.min() * 1.5, "晚间（18~21 点）的残差波动明显更大"
assert abs(t1) < 6 and abs(t24) < 20
assert abs(res1.mean()) / Y_test.mean() < 0.02, "偏差占负荷不到 2% —— 统计显著但实际不重要"
assert abs(t24) > abs(t1), "步长越长，同一份样本量下的偏差越容易被检出"

## 自查清单（能不看笔记答出来才算过）

- [ ] 10 个代码块的 assert 全部通过
- [ ] `MAPE` 的分母是什么？为什么它会「爆」？本数据实测涨了多少倍？
- [ ] `sMAPE` 和 `WAPE` 分别怎么修分母？各自的上界是多少？
- [ ] `RMSE ≥ MAE` 为什么恒成立？`RMSE/MAE = 1.2867` 说明误差是什么形状？
- [ ] 训练循环里 `clip_grad_norm_` 放在哪两步之间？它的返回值是什么？
- [ ] `sched.step()` 和 `opt.step()` 谁先谁后？
- [ ] `best_epoch` 与 `stopped_epoch` 的恒等式是什么？为什么成立？
- [ ] 递归预测把预测值接回窗口前，必须做哪一步变换？
- [ ] 递归误差方差的放大指数实测是多少？为什么远小于 0.5？
- [ ] 直接多输出一定比递归好吗？给出本章的实测数字。
- [ ] 手写 ACF 与 `Series.autocorr` 的分母差在哪？
- [ ] h=1 残差与 h=24 残差的 ACF 差多少？这说明什么？
- [ ] 「统计显著」和「实际重要」在本章分别对应哪两个数字？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 样本数 / 切分点 / 训练 / 测试 | 8569 / 6855 / 6855 / 1714 |
| 训练段目标均值 / 测试段目标均值 | 452.674164 / 483.2289 |
| 测试段时段 | 2025-10-20 15:00 ~ 2025-12-31 00:00 |
| MSE / MAE / RMSE | 1802.744577 / 32.998794 / 42.458740 |
| MAPE / sMAPE / WAPE（季节朴素） | 6.693991 / 6.682859 / 6.828811 |
| `RMSE / MAE` | 1.286676 |
| 零点陷阱后 MAE / RMSE | 143.890302 / 243.244359 |
| 零点陷阱后 MAPE / sMAPE / WAPE | 2481.953943 / 54.016410 / 39.555870 |
| MAPE 放大倍数 | 370.8× |
| 单点（0.05 vs 5.0）MAPE / sMAPE | 9900.0000% / 196.0396% |
| 早停：best_epoch / stopped_epoch / patience | 14 / 19 / 5 |
| 训练损失首末（最后一轮 0.1084） | 1.1775 → 0.1084 |
| 验证损失首末（最低 0.104366） | 0.6464 → 0.1099 |
| 梯度范数最大值（阈值 1.0）/ 被裁轮数 | 3.485 / 7 |
| 学习率 首 / 末 | 0.002 → 0.0005 |
| 递归 24 步 MAE（h1 / h24） | 17.266384（14.952325 / 19.115711） |
| 直接多输出 24 步 MAE（h1 / h24） | 17.035627（15.176231 / 18.490346） |
| 季节朴素 24 步 MAE（h1 / h24） | 32.998794（33.0215 / 33.1928） |
| 递归误差 std：h1 / h8 / h24 | 11.5343 / 13.6496 / 15.0208 |
| 方差放大幂律指数 | h^0.0861 |
| 朴素残差 ACF(1) / ACF(24) | +0.3804 / −0.3149 |
| h=1 残差 ACF(1/2/3/24/168) | +0.1543 / +0.1111 / +0.0218 / −0.0011 / +0.1646 |
| h=24 残差 ACF(1/2/3/24/168) | +0.5439 / +0.4796 / +0.3996 / −0.0617 / +0.4398 |
| h=1 残差 均值 / 占比 / t | −1.0629 kW / −0.22% / −2.3333 |
| h=24 残差 均值 / 占比 / t | −2.1429 kW / −0.44% / −3.6624 |
| 按小时残差均值 极值（小时） | −6.5399（4 点）/ +2.9676（10 点） |
| 按小时残差 std 极值（小时） | 13.7081（13 点）/ 26.7899（20 点） |